# 획 추출 → 원본 획 변형 확인 (역확산 없음)

1 Drive·기존 오류 이미지·정확히 대응하는 best.pt → 2 네 칸의 획/모델 flow 추출 → 3 기준 획 변형·정상 후보 확인.
기존 모델의 softmax 마스크와 learned flow hints를 사용합니다. 새 모델 학습과 확산 모델 다운로드는 없습니다.
직선은 직선으로, 곡선은 기준 구조를 유지하는 부드러운 변형으로 맞춥니다. 짧은 가지·작은 분리 문양을 따라가지 않도록 주 경로를 사용합니다.
폐곡선/복잡한 분기는 모양을 잘라버리는 대신 제한된 affine으로 대체하고 경고합니다. 모든 글자를 정상 복원한다고 보장하지 않습니다.
후보는 후속 역확산의 기준 후보일 뿐 실제 역확산 출력이나 정답 이미지가 아닙니다. 이 노트북은 변형의 타당성을 먼저 확인하는 단계입니다.
Drive에는 원본 ZIP 또는 원래 실험 폴더와 기존 획 추출기가 필요합니다. 결과 ZIP을 Drive에도 저장합니다.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess,sys,tempfile,json,hashlib,gc
from google.colab import drive
import os
FORCE_REMOUNT=False  # 연결이 깨진 기존 mount를 다시 연결할 때만 True
if FORCE_REMOUNT or not os.path.ismount('/content/drive'):
    print('Google Drive 연결을 승인하세요. 기존 학습 자료가 있는 계정을 선택하세요.',flush=True)
    try:
        drive.mount('/content/drive',force_remount=FORCE_REMOUNT,timeout_ms=180000)
    except Exception as exc:
        raise RuntimeError('Drive 인증/마운트에 실패했습니다. 위 Google 오류를 확인한 뒤 1번 셀을 다시 실행하세요.') from exc
print('Drive mount 확인 완료',flush=True)
root=Path(tempfile.mkdtemp(prefix='saved_repair_',dir='/content'))
subprocess.run(['git','clone','--depth','1','https://github.com/yechan25/hsqm.git',str(root)],check=True)
commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','pip','install','--prefer-binary',
    'transformers==4.57.6','huggingface-hub==0.36.0',
    'accelerate','safetensors','timm','pandas','numpy','matplotlib','pillow',
    'scipy','scikit-image','pyyaml','fonttools','ipywidgets'],check=True)
runtime_helpers={'diffusion_model/saved_repair_demo.py': '"""Load an actual saved baseline and its recorded reference conditions, not a new generation."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport zipfile\nimport numpy as np\nfrom PIL import Image\n\n\ndef pack_saved_generation(report_path, output):\n    """Package an existing Drive experiment without its original dataset/CSV."""\n    from stroke_model.convert_reference_csv import existing_path\n    report_path=existing_path(report_path)\n    report=json.loads(report_path.read_text(encoding=\'utf-8\'))\n    output=Path(output)\n    if output.exists():raise FileExistsError(output)\n    pages=[p for p in report[\'pages\'] if p[\'method\']==\'baseline\']\n    rows=[r for r in report[\'results\'] if r[\'method\']==\'reference\']\n    if len(pages)!=1 or len(rows)!=4:raise ValueError(\'Expected one baseline and four recorded reference slots\')\n    def locate(recorded):\n        basename=PurePosixPath(str(recorded).replace(\'\\\\\',\'/\')).name\n        local=report_path.parent/basename\n        return existing_path(local if local.is_file() else recorded)\n    source=locate(pages[0][\'input\'])\n    references=[locate(r[\'npz\']) for r in rows]\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(output,\'w\',zipfile.ZIP_DEFLATED) as z:\n        z.write(source,\'generation/\'+source.name)\n        for path in references:z.write(path,\'generation/\'+path.name)\n        z.writestr(\'generation/generation_report.json\',json.dumps(report,ensure_ascii=False))\n    return output\n\n\ndef prepare_saved_demo(archive, output):\n    destination=Path(output)\n    if destination.exists():raise FileExistsError(destination)\n    with zipfile.ZipFile(archive) as z:\n        names=z.namelist()\n        reports=[n for n in names if n.endswith(\'generation_report.json\')]\n        if len(reports)!=1:raise ValueError(\'ZIP must contain exactly one generation_report.json\')\n        report=json.loads(z.read(reports[0]))\n        def member(recorded):\n            basename=PurePosixPath(str(recorded).replace(\'\\\\\',\'/\')).name\n            matches=[n for n in names if PurePosixPath(n).name==basename]\n            if len(matches)!=1:raise ValueError(f\'Missing/ambiguous ZIP file: {basename}\')\n            return matches[0]\n        baseline=[p for p in report[\'pages\'] if p[\'method\']==\'baseline\']\n        if len(baseline)!=1:raise ValueError(\'A single recorded baseline page is required\')\n        source_member=member(baseline[0][\'input\'])\n        source_bytes=z.read(source_member)\n        with Image.open(io.BytesIO(source_bytes)) as image:\n            width,height=image.size\n        if width!=height or width%8:raise ValueError(\'Expected a square saved sheet, size divisible by 8\')\n        # Reference arrays are metadata conditions AFTER generation, never the repair input.\n        templates=[];boxes=[]\n        for index in range(4):\n            candidates=[r for r in report[\'results\'] if r[\'method\']==\'reference\' and r[\'slot\']==index]\n            if len(candidates)!=1:raise ValueError(\'Four reference slots required\')\n            row=candidates[0];boxes.append(row[\'crop_bbox\'])\n            try:npz_member=member(row[\'npz\'])\n            except ValueError:npz_member=member(f\'reference_slot_{index+1}.npz\')\n            with np.load(io.BytesIO(z.read(npz_member)),allow_pickle=False) as data:\n                ref=data[\'reference\'].astype(np.float32)\n                strokes=data[\'reference_strokes\'].astype(np.float32)\n            if ref.ndim!=3 or strokes.ndim!=3 or ref.shape[0]!=1 or ref.shape[1:]!=strokes.shape[1:]:\n                raise ValueError(\'Malformed saved reference conditions\')\n            templates.append((ref,strokes))\n    destination.mkdir(parents=True)\n    source=destination/\'saved_error_input.png\';source.write_bytes(source_bytes)\n    manifest=[]\n    for i,(reference,strokes) in enumerate(templates):\n        name=f\'condition_{i+1}.npz\'\n        np.savez_compressed(destination/name,reference=reference,strokes=strokes)\n        manifest.append(dict(path=name,split=\'train\',char_id=\'\'))\n    path=destination/\'conditions.json\';path.write_text(json.dumps(manifest),encoding=\'utf-8\')\n    provenance=dict(source_archive=str(archive),source_member=source_member,\n        source_sha256=hashlib.sha256(source_bytes).hexdigest(),\n        source_initialization=report.get(\'init_mode\',\'not_recorded_in_legacy_report\'),\n        source_prompt=report.get(\'prompt\'),source_report=report,\n        demonstration=\'reference_conditioned_repair_of_saved_baseline\',\n        fresh_text_to_image=False,reference_encoded_as_repair_input=False)\n    (destination/\'source_provenance.json\').write_text(json.dumps(provenance,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    return dict(source=str(source),manifest=str(path),rows=[1,2,3,4],boxes=boxes,\n        size=width,checkpoint=report[\'checkpoint\'],checkpoint_sha256=report.get(\'checkpoint_sha256\'),\n        provenance=provenance)\n', 'stroke_model/stroke_reference_fit.py': '"""Fit valid reference strokes to extracted stroke positions, not their defects.\n\nInference-only geometry: principal skeleton paths discard short side branches;\nstraight references remain straight, curves receive a regularized deformation.\nThe original reference centerline is deformed and re-rendered at the observed\nwidth. Output is a candidate for inspection, not a ground-truth reconstruction.\n"""\nfrom dataclasses import dataclass, asdict\nimport heapq\nimport numpy as np\nfrom scipy.ndimage import label, distance_transform_edt, map_coordinates, affine_transform,gaussian_filter1d\nfrom scipy.spatial import cKDTree\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass ReferenceFitConfig:\n    points: int=64\n    smoothness: float=12.\n    shape_prior: float=.15\n    max_displacement: float=16.\n    max_width: float=10.\n    min_width: float=1.\n    min_length_ratio: float=.65\n    max_length_ratio: float=1.6\n    junction_radius: float=2.5\n    junction_weight: float=8.\n    max_junction_shift: float=2.\n\n    def __post_init__(self):\n        if type(self.points) is not int or self.points<8:raise ValueError(\'At least eight path samples required\')\n        if any(not np.isfinite(v) or v<0 for v in asdict(self).values()):raise ValueError(\'Invalid fit configuration\')\n        if not self.min_width or self.max_width<self.min_width or not 0<self.min_length_ratio<self.max_length_ratio:\n            raise ValueError(\'Invalid width/length limits\')\n\n\ndef _unit(array):\n    array=np.asarray(array,np.float32)\n    if array.ndim!=2 or not np.isfinite(array).all() or array.min()<0 or array.max()>1:\n        raise ValueError(\'Expected a finite 2D ink mask in [0,1]\')\n    return array\n\n\ndef _component(mask,hint=None):\n    threshold=max(.1,min(.4,float(mask.max())*.4))\n    binary=mask>=threshold\n    labels,count=label(binary,np.ones((3,3)))\n    if not count:raise ValueError(\'Empty/low-confidence extracted stroke\')\n    distance=None\n    if hint is not None and hint.max()>.1:\n        distance=distance_transform_edt(hint<max(.1,float(hint.max())*.3))\n    scores=[]\n    for k in range(1,count+1):\n        region=labels==k;mass=float(mask[region].sum())\n        proximity=1. if distance is None else np.exp(-float(np.median(distance[region]))/20.)\n        scores.append(mass*proximity)\n    chosen=labels==(int(np.argmax(scores))+1)\n    return chosen,dict(components=count,selected_mass_fraction=float(mask[chosen].sum()/max(float(mask.sum()),1e-6)))\n\n\ndef _resample(path,n):\n    arc=np.r_[0,np.cumsum(np.linalg.norm(np.diff(path,axis=0),axis=1))]\n    if arc[-1]<1e-6:raise ValueError(\'Degenerate stroke path\')\n    keep=np.r_[True,np.diff(arc)>1e-8];arc=arc[keep];path=path[keep]\n    t=np.linspace(0,arc[-1],n)\n    return np.stack([np.interp(t,arc,path[:,i]) for i in range(2)],1)\n\n\ndef principal_path(binary,n=64):\n    """Longest geodesic path of the principal component; never reject spurs."""\n    skeleton=skeletonize(binary)\n    coordinates=np.argwhere(skeleton)\n    if len(coordinates)<2:raise ValueError(\'Insufficient centerline support\')\n    indices={tuple(yx):i for i,yx in enumerate(coordinates)}\n    graph=[[] for _ in coordinates]\n    for i,(y,x) in enumerate(coordinates):\n        for dy in (-1,0,1):\n            for dx in (-1,0,1):\n                if not (dx or dy):continue\n                j=indices.get((y+dy,x+dx))\n                if j is not None:graph[i].append((j,float(np.hypot(dx,dy))))\n    endpoints=[i for i,edges in enumerate(graph) if len(edges)==1]\n    def farthest(start):\n        distances=np.full(len(graph),np.inf);distances[start]=0.;parent=np.full(len(graph),-1,int)\n        queue=[(0.,start)]\n        while queue:\n            cost,i=heapq.heappop(queue)\n            if cost>distances[i]+1e-8:continue\n            for j,weight in graph[i]:\n                proposed=cost+weight\n                if proposed<distances[j]-1e-8:\n                    distances[j]=proposed;parent[j]=i;heapq.heappush(queue,(proposed,j))\n        candidates=endpoints if endpoints else list(range(len(graph)))\n        end=max(candidates,key=lambda j:distances[j] if np.isfinite(distances[j]) else -1)\n        return end,parent\n    first,_=farthest(endpoints[0] if endpoints else 0)\n    last,parent=farthest(first);order=[last]\n    while order[-1]!=first:\n        previous=parent[order[-1]]\n        if previous<0:raise ValueError(\'Disconnected centerline\')\n        order.append(int(previous))\n    path=coordinates[order,::-1].astype(float)\n    return _resample(path,n),dict(endpoints=len(endpoints),path_coverage=len(order)/len(coordinates),closed=len(endpoints)==0)\n\n\ndef _width(binary,path):\n    radius=distance_transform_edt(binary)\n    samples=map_coordinates(radius,[path[:,1],path[:,0]],order=1,mode=\'nearest\')\n    middle=samples[len(samples)//10:len(samples)-len(samples)//10]\n    return max(1.,float(np.median(2*middle-1)))\n\n\ndef render_reference(reference,original_path,target_path,width_scale):\n    """Render the deformed original centerline with the estimated stroke width.\n\n    Raster thin hints must not be stretched directly: their one-pixel staircase\n    and interpolation profile create serrated, gray thick strokes.\n    """\n    reference=_unit(reference);h,w=reference.shape\n    # Dense samples reduce nearest-segment seams while preserving reference caps.\n    # Keep normalized arc correspondence even when relative segment lengths change.\n    target=np.stack([np.interp(np.linspace(0,1,512),np.linspace(0,1,len(target_path)),target_path[:,i]) for i in range(2)],1)\n    yy,xx=np.mgrid[:h,:w];pixels=np.stack((xx.ravel(),yy.ravel()),1)\n    binary=reference>=max(.1,float(reference.max())*.3)\n    width=_width(binary,original_path)*width_scale\n    distance,_=cKDTree(target).query(pixels)\n    return np.clip(.5+(width/2-distance),0,1).reshape(h,w).astype(np.float32)\n\n\ndef _affine_fallback(reference,observed,config):\n    """Preserve a loop/branch prototype rather than cutting it into a path."""\n    def moments(mask):\n        y,x=np.nonzero(mask);xy=np.stack((x,y),1).astype(float)\n        center=xy.mean(0);extent=np.percentile(xy,95,axis=0)-np.percentile(xy,5,axis=0)\n        return center,np.maximum(extent,1)\n    source=reference>max(.1,float(reference.max())*.3)\n    rc,re=moments(source);tc,te=moments(observed)\n    shift=tc-rc;shift*=min(1.,config.max_displacement/max(np.linalg.norm(shift),1e-6))\n    scale=np.clip(te/re,.75,1.35)\n    # scipy uses y,x and maps output to input.\n    inverse=np.diag(1/scale[::-1]);offset=rc[::-1]-inverse@(rc+shift)[::-1]\n    result=affine_transform(reference,inverse,offset=offset,output_shape=reference.shape,order=1,mode=\'constant\')\n    return result.clip(0,1).astype(np.float32),dict(translation=shift.tolist(),scale=scale.tolist())\n\n\ndef fit_reference_stroke(reference,extracted,*,hint=None,config=None):\n    config=config or ReferenceFitConfig();reference=_unit(reference);extracted=_unit(extracted)\n    if reference.shape!=extracted.shape:raise ValueError(\'Reference/extraction shape mismatch\')\n    hint=reference if hint is None else _unit(hint)\n    if hint.shape!=reference.shape:raise ValueError(\'Hint shape mismatch\')\n    observed,component_info=_component(extracted,hint)\n    canonical,_=_component(reference)\n    original,path_info=principal_path(canonical,config.points)\n    metadata=dict(component_info,reference_path=path_info,warnings=[])\n    if path_info[\'closed\'] or path_info[\'path_coverage\']<.65 or (path_info[\'endpoints\']>2 and path_info[\'path_coverage\']<.9):\n        warped,transform=_affine_fallback(reference,observed,config)\n        metadata.update(method=\'affine_topology_preserving_fallback\',transform=transform)\n        metadata[\'warnings\'].append(\'Closed/branched prototype: affine fallback, no open-curve fit\')\n        return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n            reference_path=None,target_path=None,hint_path=None,width_scale=1.,metadata=metadata)\n    try:\n        hint_binary,_=_component(hint)\n        initial,_=principal_path(hint_binary,config.points)\n        if np.linalg.norm(initial[[0,-1]]-original[[0,-1]])>np.linalg.norm(initial[::-1][[0,-1]]-original[[0,-1]]):initial=initial[::-1]\n        if np.max(np.linalg.norm(initial-original,axis=1))>config.max_displacement:\n            initial=original.copy();metadata[\'warnings\'].append(\'Learned hint displacement too large; initialized from reference\')\n    except ValueError:\n        initial=original.copy();metadata[\'warnings\'].append(\'No usable learned hint; initialized from reference\')\n    target,target_info=principal_path(observed,config.points)\n    if np.linalg.norm(target[[0,-1]]-initial[[0,-1]])>np.linalg.norm(target[::-1][[0,-1]]-initial[[0,-1]]):target=target[::-1]\n    metadata[\'extracted_path\']=target_info\n    if target_info[\'closed\']:\n        target=initial.copy();metadata[\'warnings\'].append(\'Closed extracted path for open reference: retained initialization\')\n    if target_info[\'path_coverage\']<.75:metadata[\'warnings\'].append(\'Side branches excluded by principal-path fit\')\n    if component_info[\'selected_mass_fraction\']<.65:metadata[\'warnings\'].append(\'Large disconnected extraction mass excluded\')\n    chord=original[-1]-original[0];length=np.linalg.norm(np.diff(original,axis=0),axis=1).sum()\n    normal=np.array([-chord[1],chord[0]])/max(np.linalg.norm(chord),1e-6)\n    deviation=np.max(np.abs((original-original[0])@normal))\n    straight=deviation<=1.1 or deviation/max(length,1)<.025\n    if straight:\n        # Robust TLS: trim off-axis centerline points before estimating direction.\n        points=target.copy()\n        for _ in range(3):\n            center=points.mean(0);_,_,v=np.linalg.svd(points-center,full_matrices=False);axis=v[0]\n            residual=np.abs((target-center)@np.array([-axis[1],axis[0]]))\n            points=target[residual<=max(.75,float(np.quantile(residual,.85)))]\n        projection=(target-center)@axis\n        if axis@(initial[-1]-initial[0])<0:axis=-axis;projection=-projection\n        endpoints=np.quantile(projection,[.01,.99])\n        target=center+np.linspace(*endpoints,config.points)[:,None]*axis\n        metadata[\'method\']=\'robust_straight_reference_fit\'\n    else:metadata[\'method\']=\'regularized_curved_reference_fit\'\n    target_length=np.linalg.norm(np.diff(target,axis=0),axis=1).sum()\n    ratio=target_length/max(length,1e-6)\n    if not config.min_length_ratio<=ratio<=config.max_length_ratio:\n        metadata[\'warnings\'].append(\'Extracted path length outside allowed range; retained bounded initialization\')\n        target=initial.copy()\n        if straight:target=np.linspace(target[0],target[-1],config.points)\n    n=config.points;second=np.diff(np.eye(n),n=2,axis=0)\n    system=(1+config.shape_prior)*np.eye(n)+config.smoothness*(second.T@second)\n    if straight:\n        fitted=target\n    else:\n        # Smooth the deformation, rather than tracing every observed bend/spur.\n        fitted=initial+np.linalg.solve(system,target-initial)\n        endpoints=fitted[[0,-1]].copy()\n        fitted=gaussian_filter1d(fitted,.8,axis=0,mode=\'nearest\')\n        fitted[0],fitted[-1]=endpoints\n    displacement=fitted-original\n    norm=np.linalg.norm(displacement,axis=1,keepdims=True)\n    limit=min(1.,config.max_displacement/max(float(norm.max()),1e-6)) if straight else np.minimum(1,config.max_displacement/np.maximum(norm,1e-6))\n    fitted=original+displacement*limit\n    ref_width=_width(canonical,original)\n    target_width=float(np.clip(_width(observed,target),config.min_width,config.max_width))\n    width_scale=float(np.clip(target_width/ref_width,.5,8.))\n    warped=render_reference(reference,original,fitted,width_scale)\n    metadata.update(width_reference=ref_width,width_observed=target_width,width_scale=width_scale,\n        extracted_length_ratio=float(ratio),max_displacement=float(np.linalg.norm(fitted-original,axis=1).max()))\n    return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n        reference_path=original,target_path=fitted,hint_path=initial,width_scale=width_scale,metadata=metadata)\n\n\ndef fit_reference_glyph(strokes,extracted,*,hints=None,glyph_reference=None,config=None):\n    config=config or ReferenceFitConfig()\n    strokes=np.asarray(strokes,np.float32);extracted=np.asarray(extracted,np.float32)\n    if strokes.shape!=extracted.shape or strokes.ndim!=3:raise ValueError(\'K matching reference/extracted masks required\')\n    hints=strokes if hints is None else np.asarray(hints,np.float32)\n    if hints.shape!=strokes.shape:raise ValueError(\'Hint shape mismatch\')\n    fitted=[]\n    for k in range(len(strokes)):\n        try:item=fit_reference_stroke(strokes[k],extracted[k],hint=hints[k],config=config)\n        except ValueError as exc:\n            item=dict(warped=strokes[k].copy(),hint=hints[k],reference=strokes[k],extracted=extracted[k],\n                reference_path=None,target_path=None,hint_path=None,width_scale=1.,\n                metadata=dict(method=\'unreliable_reference_fallback\',warnings=[str(exc)]))\n        fitted.append(item)\n    # Joint bounded translations keep reference contacts from independently drifting.\n    matrix=np.eye(len(strokes));rhs=np.zeros((len(strokes),2));contacts=[]\n    for i,a in enumerate(fitted):\n        if a[\'reference_path\'] is None:continue\n        for j in range(i+1,len(fitted)):\n            b=fitted[j]\n            if b[\'reference_path\'] is None:continue\n            distance=np.linalg.norm(a[\'reference_path\'][:,None]-b[\'reference_path\'][None],axis=2)\n            u,v=np.unravel_index(distance.argmin(),distance.shape)\n            connected=distance[u,v]<=config.junction_radius\n            if not connected and glyph_reference is not None and distance[u,v]<=10:\n                line=np.linspace(a[\'reference_path\'][u],b[\'reference_path\'][v],32)\n                ink=map_coordinates(_unit(np.asarray(glyph_reference).squeeze()),[line[:,1],line[:,0]],order=1,mode=\'constant\')\n                connected=float((ink>.25).mean())>.85\n            if not connected:continue\n            # Sparse annotations often omit the segment inside a junction. If\n            # the full reference confirms a connection, extend an endpoint to\n            # the other stroke instead of translating its whole visible tip.\n            if glyph_reference is not None:\n                ia=0 if u<4 else -1 if u>=len(a[\'reference_path\'])-4 else None\n                ib=0 if v<4 else -1 if v>=len(b[\'reference_path\'])-4 else None\n                if (ia is None)!=(ib is None):\n                    end_item,end_index,other,other_index=(a,ia,b,v) if ia is not None else (b,ib,a,u)\n                    difference=other[\'target_path\'][other_index]-end_item[\'target_path\'][end_index]\n                    gap=float(np.linalg.norm(difference))\n                    if gap<=config.max_displacement:\n                        t=np.linspace(0,1,len(end_item[\'target_path\']))\n                        factor=(1-t)**3 if end_index==0 else t**3\n                        if end_item[\'metadata\'].get(\'method\')==\'robust_straight_reference_fit\':factor=(1-t) if end_index==0 else t\n                        end_item[\'target_path\']=end_item[\'target_path\']+factor[:,None]*difference\n                        end_item[\'metadata\'].setdefault(\'endpoint_extensions\',[]).append(gap)\n            difference=a[\'target_path\'][u]-b[\'target_path\'][v]\n            weight=config.junction_weight\n            matrix[i,i]+=weight;matrix[j,j]+=weight;matrix[i,j]-=weight;matrix[j,i]-=weight\n            rhs[i]-=weight*difference;rhs[j]+=weight*difference\n            contacts.append(dict(strokes=[i,j],indices=[int(u),int(v)],gap_before=float(np.linalg.norm(difference))))\n    shift=np.linalg.solve(matrix,rhs)\n    shift*=np.minimum(1,config.max_junction_shift/np.maximum(np.linalg.norm(shift,axis=1,keepdims=True),1e-6))\n    for i,item in enumerate(fitted):\n        if item[\'target_path\'] is None:continue\n        item[\'target_path\']=item[\'target_path\']+shift[i]\n        displacement=item[\'target_path\']-item[\'reference_path\']\n        maximum=float(np.linalg.norm(displacement,axis=1).max())\n        if maximum>config.max_displacement:\n            item[\'target_path\']=item[\'reference_path\']+displacement*(config.max_displacement/maximum)\n            item[\'metadata\'][\'warnings\'].append(\'Joint displacement clipped; inspect contact consistency\')\n        item[\'warped\']=render_reference(item[\'reference\'],item[\'reference_path\'],item[\'target_path\'],item[\'width_scale\'])\n        item[\'metadata\'][\'junction_translation\']=shift[i].tolist()\n        item[\'metadata\'][\'final_max_displacement\']=float(np.linalg.norm(item[\'target_path\']-item[\'reference_path\'],axis=1).max())\n    for contact in contacts:\n        i,j=contact[\'strokes\'];u,v=contact[\'indices\']\n        contact[\'gap_after\']=float(np.linalg.norm(fitted[i][\'target_path\'][u]-fitted[j][\'target_path\'][v]))\n    masks=np.stack([item[\'warped\'] for item in fitted])\n    return dict(strokes=masks,candidate=masks.max(0),fits=fitted,contacts=contacts,config=asdict(config))\n'}
for name,code in runtime_helpers.items():
    (root/name).write_text(code,encoding='utf-8')
sys.path.insert(0,str(root))
for name in list(sys.modules):
    if name.split('.')[0] in ('stroke_model','diffusion_model'):del sys.modules[name]
import torch,numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
if not torch.cuda.is_available():raise RuntimeError('L4 GPU 런타임을 선택하세요.')
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.saved_repair_demo import prepare_saved_demo
import shutil
from diffusion_model.saved_repair_demo import pack_saved_generation
HSQM_PATH='/content/drive/Shareddrives/2026 자율연구/HSQM'
HSQM=existing_path(HSQM_PATH)
ARCHIVE=None  # Drive ZIP 경로를 명시할 수 있음. None이면 아래 위치/원래 생성 폴더 확인
CHECKPOINT=None  # None이면 당시 기록된 checkpoint 경로. 최신 모델 임의 선택 없음
SOURCE_REPORT=HSQM/'topo_guided_diffusion/20261108_103932_929716/generation_report.json'
if ARCHIVE is None:
    candidates=[HSQM/'guidance_debug (4).zip',HSQM/'debug/guidance_debug (4).zip',
        Path('/content/drive/MyDrive/guidance_debug (4).zip'),
        Path('/content/drive/MyDrive/Downloads/guidance_debug (4).zip')]
    ARCHIVE=next((p for p in candidates if p.is_file()),None)
if ARCHIVE is None:
    try:report_path=existing_path(SOURCE_REPORT)
    except FileNotFoundError:report_path=None
    if report_path is not None:
        print('Drive의 기존 생성 실험에서 입력을 복원합니다:',report_path,flush=True)
        try:ARCHIVE=pack_saved_generation(report_path,root/'source_experiment.zip')
        except FileNotFoundError as exc:print('기존 생성 폴더의 파일이 부족합니다:',exc)
if ARCHIVE is None:
    print('기존 ZIP/생성 폴더를 찾지 못했습니다. guidance_debug (4).zip만 업로드하세요.')
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('ZIP 하나를 선택하세요.')
    ARCHIVE=Path(next(iter(uploaded)))
archive_origin=str(ARCHIVE)
local_archive=root/'saved_input.zip'
shutil.copyfile(existing_path(ARCHIVE),local_archive)
RUN_DIR=Path('/content/stroke_reference_fit_runs')/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
demo=prepare_saved_demo(local_archive,RUN_DIR/'input')
if CHECKPOINT is None:
    try:CHECKPOINT=existing_path(demo['checkpoint'])
    except FileNotFoundError:
        CHECKPOINT=HSQM/'v2_topology_runs/20261104_214705_208224/best.pt'
checkpoint_origin=existing_path(CHECKPOINT)
CHECKPOINT=root/'extractor_best.pt'
print('Drive 획 모델을 실행용 임시 저장소로 복사합니다:',checkpoint_origin,flush=True)
shutil.copyfile(checkpoint_origin,CHECKPOINT)
actual_sha=hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
if demo['checkpoint_sha256'] and actual_sha!=demo['checkpoint_sha256']:
    raise ValueError('ZIP 당시 획 모델과 SHA256이 다릅니다. CHECKPOINT를 확인하세요.')
(RUN_DIR/'drive_sources.json').write_text(json.dumps(dict(archive=archive_origin,checkpoint=str(checkpoint_origin)),ensure_ascii=False,indent=2))
print('GPU:',torch.cuda.get_device_name(0),'| 실제 입력:',demo['source'])
print('획 모델:',checkpoint_origin,'| SHA256:',actual_sha)
display(Image.open(demo['source']))

In [ ]:
from diffusion_model.score_pilot import training_catalog
from diffusion_model.multiglyph_pilot import compose_sheet
from stroke_model.infer_topology import load_topology
from torch.nn import functional as F

extractor,size,epoch=load_topology(CHECKPOINT,torch.device('cuda'))
extractor.requires_grad_(False)
_,slots=compose_sheet(training_catalog(demo['manifest']),demo['size'])
for slot,box in zip(slots,demo['boxes']):slot['bbox']=box
with Image.open(demo['source']) as image:source_array=np.asarray(image.convert('RGB'),np.float32)/255
source_tensor=torch.from_numpy(source_array.copy()).permute(2,0,1)[None].cuda()
extractions=[]
for i,slot in enumerate(slots):
    x0,y0,x1,y1=slot['bbox']
    reference=torch.as_tensor(slot['reference'],device='cuda',dtype=torch.float32)[None]
    strokes=torch.as_tensor(slot['strokes'],device='cuda',dtype=torch.float32)[None]
    if reference.shape[-2:]!=(size,size):raise ValueError('Checkpoint/reference resolution mismatch')
    crop=F.interpolate(source_tensor[...,y0:y1,x0:x1],(size,size),mode='bilinear',align_corners=False)
    ink=1-(crop*crop.new_tensor([.299,.587,.114])[None,:,None,None]).sum(1,keepdim=True)
    with torch.no_grad():output=extractor(ink,reference,strokes)
    data=dict(reference=reference[0,0].cpu().numpy(),strokes=strokes[0].cpu().numpy(),
        ink=ink[0,0].cpu().numpy(),probabilities=output['probabilities'][0].cpu().numpy(),
        masks=output['masks'][0].cpu().numpy(),
        hints=output.get('hints',strokes)[0].cpu().numpy())
    if 'flow' in output:data['flow']=output['flow'][0].cpu().numpy()
    data['hint_from_model']=np.array('hints' in output)
    extractions.append(data)
    np.savez_compressed(RUN_DIR/f'extraction_slot_{i+1}.npz',**data)
    fig,axes=plt.subplots(3,len(strokes[0]),figsize=(3*len(strokes[0]),7),squeeze=False)
    for k in range(len(strokes[0])):
        for row,(key,title) in enumerate([('strokes','Original reference'),('hints','Model flow warp'),('masks','Extracted ink')]):
            axes[row,k].imshow(data[key][k],cmap='gray',vmin=0,vmax=1);axes[row,k].set_title(f'{title} {k+1}');axes[row,k].axis('off')
    fig.suptitle(f'Slot {i+1}');fig.tight_layout();fig.savefig(RUN_DIR/f'extraction_slot_{i+1}.png',dpi=120);plt.show()
extractor.cpu();gc.collect();torch.cuda.empty_cache()
print('네 칸의 추출을 저장했습니다. 3번은 CPU에서 기준 획을 변형합니다.')

In [ ]:
from stroke_model.stroke_reference_fit import ReferenceFitConfig,fit_reference_glyph
from scipy.ndimage import distance_transform_edt
import pandas as pd
import zipfile

CONFIG=ReferenceFitConfig(max_displacement=16.,smoothness=12.,shape_prior=.15,max_junction_shift=2.)
RUN_GEOMETRY_CONTROL=True
results=[];controls=[];summary=[]
for i,data in enumerate(extractions):
    result=fit_reference_glyph(data['strokes'],data['masks'],hints=data['hints'],glyph_reference=data['reference'],config=CONFIG)
    results.append(result)
    # Strong reference-only correspondence control, using exactly the same ink.
    distance=np.stack([distance_transform_edt(s<.2) for s in data['strokes']])
    ownership=distance.argmin(0)
    nearest=np.stack([data['ink']*(ownership==k) for k in range(len(distance))])
    control=fit_reference_glyph(data['strokes'],nearest,hints=data['strokes'],glyph_reference=data['reference'],config=CONFIG) if RUN_GEOMETRY_CONTROL else None
    controls.append(control)
    arrays=dict(warped_strokes=result['strokes'],candidate_ink=result['candidate'],
        reference_strokes=data['strokes'],extracted_masks=data['masks'],model_hints=data['hints'])
    if control is not None:arrays['geometry_candidate_ink']=control['candidate']
    np.savez_compressed(RUN_DIR/f'warped_slot_{i+1}.npz',**arrays)
    fig,axes=plt.subplots(len(result['fits']),5,figsize=(15,2.7*len(result['fits'])),squeeze=False)
    for k,item in enumerate(result['fits']):
        observed=data['masks'][k];fitted=item['warped']
        overlap=np.stack((observed,np.minimum(observed,fitted),fitted),-1).clip(0,1)
        for ax,(image,title) in zip(axes[k],[(data['strokes'][k],'Original'),(data['hints'][k],'Learned warp'),
            (observed,'Extracted'),(fitted,'Fitted reference'),(overlap,'Red=input / blue=candidate')]):
            ax.imshow(image,cmap='gray',vmin=0,vmax=1);ax.set_title(f'{k+1}: {title}');ax.axis('off')
        if item['reference_path'] is not None:
            rp=item['reference_path'];tp=item['target_path']
            axes[k,3].plot(tp[:,0],tp[:,1],color='cyan',linewidth=.6)
        info=dict(slot=i+1,stroke=k+1,**item['metadata'])
        summary.append(info)
    fig.suptitle(f'Slot {i+1}: inspect every fitted stroke');fig.tight_layout();fig.savefig(RUN_DIR/f'fit_slot_{i+1}.png',dpi=120);plt.show()
    (RUN_DIR/f'fit_slot_{i+1}.json').write_text(json.dumps(dict(strokes=[x['metadata'] for x in result['fits']],contacts=result['contacts']),ensure_ascii=False,indent=2))

def compose_candidate(masks_by_slot):
    canvas=Image.new('RGB',(demo['size'],demo['size']),'white')
    for slot,mask in zip(slots,masks_by_slot):
        x0,y0,x1,y1=slot['bbox'];crop=source_array[y0:y1,x0:x1]
        brightness=crop.mean(-1)
        background=np.median(crop[brightness>=np.quantile(brightness,.8)],axis=0)
        foreground=np.median(crop[brightness<=np.quantile(brightness,.05)],axis=0)
        alpha=np.asarray(Image.fromarray(mask.astype(np.float32)).resize((x1-x0,y1-y0),Image.Resampling.BICUBIC)).clip(0,1)
        rgb=background[None,None]*(1-alpha[...,None])+foreground[None,None]*alpha[...,None]
        canvas.paste(Image.fromarray(np.rint(rgb*255).clip(0,255).astype('uint8')),(x0,y0))
    return canvas
candidate=compose_candidate([r['candidate'] for r in results]);candidate.save(RUN_DIR/'normal_candidate.png')
original=compose_candidate([d['reference'] for d in extractions]);original.save(RUN_DIR/'unfitted_reference.png')
hint_candidate=compose_candidate([d['hints'].max(0) for d in extractions]);hint_candidate.save(RUN_DIR/'learned_hint_candidate.png')
panels=[('Erroneous input',Image.open(demo['source']).convert('RGB')),('Unfitted full reference',original),
    ('Model hints (thin)',hint_candidate),('Fitted normal candidate',candidate)]
if RUN_GEOMETRY_CONTROL:
    geometry_candidate=compose_candidate([r['candidate'] for r in controls]);geometry_candidate.save(RUN_DIR/'geometry_candidate.png')
    panels.append(('Without learned extraction',geometry_candidate))
fig,axes=plt.subplots(1,len(panels),figsize=(5*len(panels),5))
for ax,(title,image) in zip(axes,panels):ax.imshow(image);ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(RUN_DIR/'candidate_comparison.png',dpi=140);plt.show()
source=np.asarray(panels[0][1],np.float32)/255;target=np.asarray(candidate,np.float32)/255
source_ink=1-source.mean(-1);target_ink=1-target.mean(-1)
erase=np.maximum(source_ink-target_ink,0);restore=np.maximum(target_ink-source_ink,0)
np.savez_compressed(RUN_DIR/'candidate_difference.npz',erase=erase,restore=restore,source_ink=source_ink,candidate_ink=target_ink)
fig,axes=plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(candidate);axes[0].set_title('Candidate ONLY — not diffusion output')
axes[1].imshow(erase,cmap='magma',vmin=0,vmax=1);axes[1].set_title('Proposed erase (includes alignment error)')
axes[2].imshow(restore,cmap='magma',vmin=0,vmax=1);axes[2].set_title('Proposed restore (includes alignment error)')
for ax in axes:ax.axis('off')
fig.tight_layout();fig.savefig(RUN_DIR/'candidate_guides.png',dpi=140);plt.show()
pd.DataFrame(summary).to_csv(RUN_DIR/'stroke_fit_diagnostics.csv',index=False)
warnings=[dict(slot=r['slot'],stroke=r['stroke'],warnings=r['warnings']) for r in summary if r.get('warnings')]
display(pd.DataFrame(summary)[['slot','stroke','method','warnings']])
print('경고:',len(warnings),'개. 후보의 획 모양/접점/굵기를 확인한 뒤에만 다음 역확산 실험에 사용하세요.')
print('이 단계는 실제 역확산을 실행하지 않습니다.')
(RUN_DIR/'runtime_helpers.json').write_text(json.dumps(runtime_helpers,ensure_ascii=False))
(RUN_DIR/'git_commit.txt').write_text(commit)
archive=RUN_DIR.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.rglob('*'):
        if path.is_file():z.write(path,path.relative_to(RUN_DIR))
drive_folder=HSQM/'stroke_reference_fit_results';drive_folder.mkdir(exist_ok=True)
shutil.copy2(archive,drive_folder/archive.name)
print('Drive 결과:',drive_folder/archive.name)
files.download(str(archive))